# QQQ Top-20 → Max-Sharpe Portfolio, Rebalanced Monthly

Back-test of the following strategy against two buy-and-hold baselines.

| | |
|---|---|
| **Principal** | $10,000, invested once at the start |
| **Universe** | Nasdaq-100 constituents (what QQQ holds) |
| **Selection** | The dashboard's own screen — `screening.evaluate_ticker` — run point-in-time, ranked by `daily_annret`, top 20 |
| **Sizing** | Long-only maximum-Sharpe mean-variance weights over the selected names |
| **Rebalance** | Monthly: new selection and new weights at the first trading day of each month, held to the first trading day of the next |
| **Window** | 2025-06-01 → today |
| **Baselines** | Buy and hold QQQ; buy and hold BOXX |

## What "the same stock selection" means here

`screening.py` is the Screener tab's engine. `evaluate_ticker` applies, in order:

1. a price gate (`min_price`, $10) and a liquidity gate (`min_avg_volume`, 200k on the 50-day average),
2. Minervini Stage-2: close above both the 150- and 200-day SMA, and both of those SMAs rising over the last 20 bars,
3. within 25% of the 52-week high,
4. positive 6-month (126-bar) relative strength versus `^GSPC`.

`run_screening` then sorts everything that passes by `daily_annret` — the annualised
return over the trailing window. This notebook calls **the same function**, not a
re-implementation, so the selection cannot silently diverge from the dashboard's.

The one change is *when* it is asked. `evaluate_ticker` reads `.iloc[-1]`: it always
answers "as of the last bar you hand it". The dashboard hands it today's history; this
notebook hands it history truncated at each rebalance date, which makes the same code
point-in-time. Nothing downstream of the truncation can see the future.

## Known limitations — read these before trusting a number

* **Survivorship / membership bias.** The universe is *today's* Nasdaq-100, held
  fixed across the whole window. Names added during 2025-26 are in the pool before
  they were actually in QQQ, and names dropped are missing. Over a 15-month window
  this is small but it is not zero, and it biases the strategy's return upward.
* **Fractional shares** are assumed. With $10,000 across 20 names the average
  position is $500, and several Nasdaq-100 names trade above that, so whole-share
  rounding would distort the weights more than fractional fills distort realism.
* **No taxes.** Monthly rebalancing in a taxable account realises short-term gains.
* **Costs** are the repo's `backtest.CostModel` default — 5 bps slippage, 1 bp
  commission, one way — applied to every fill.
* **Fifteen months is not a sample.** Everything below is one path through one
  regime. It cannot distinguish skill from a good quarter for large-cap tech.

In [ ]:
from __future__ import annotations

import sys
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

# The notebook lives in notebooks/; the strategy modules live one level up.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import data as data_mod
import screening
from backtest import TRADING_DAYS, CostModel
from strategy import Action
from config import SCREENING_PARAMS

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

print(f"repo root: {REPO_ROOT}")

## 1 · Configuration

Every knob the back-test reads is here. `SCREENING_PARAMS` is imported from
`config.py` untouched, so changing the screen in the dashboard changes it here too.

In [ ]:
@dataclass(frozen=True)
class BacktestSettings:
    principal: float = 10_000.0
    start: str = "2025-06-01"
    end: str | None = None            # None = today

    top_n: int = 20
    benchmarks: tuple[str, ...] = ("QQQ", "BOXX")
    #: The screen's relative-strength benchmark. Same as the dashboard's.
    rs_benchmark: str = SCREENING_PARAMS["market_benchmark"]
    #: Trailing window handed to the screen, in bars. 3y matches SCREENING_PARAMS["period"].
    screen_window: int = 756

    #: Trailing daily returns used to estimate the covariance for the optimiser.
    cov_lookback: int = 252
    #: Below this many overlapping bars a name cannot be optimised and is dropped.
    cov_min_bars: int = 60
    #: Shrinkage of the sample covariance toward its diagonal. 0 = raw sample.
    cov_shrinkage: float = 0.10
    #: Hard cap on one name's weight. Mirrors config.MAX_POSITION_PCT.
    max_weight: float = 0.25
    #: Symbol whose trailing return stands in for the risk-free rate, or None for 0.
    rf_proxy: str | None = "BOXX"

    costs: CostModel = field(default_factory=CostModel)
    #: How much of each rebalance's equity is held back to cover slippage and commission.
    cost_buffer: float = 0.002

    #: History pulled per symbol. Must cover screen_window before `start`.
    history_period: str = "5y"


CFG = BacktestSettings()
END = pd.Timestamp(CFG.end) if CFG.end else pd.Timestamp.today().normalize()
START = pd.Timestamp(CFG.start)

CFG

## 2 · Universe — what QQQ holds

Scraped from Wikipedia the same way `screening.load_sp500_symbols` scrapes the S&P 500.
The static list is the fallback for an offline or rate-limited run; it is a snapshot,
which is where the membership bias flagged at the top comes from.

In [ ]:
# Snapshot fallback. Preferred source is the live scrape below.
NDX_STATIC = [
    "AAPL", "ABNB", "ADBE", "ADI", "ADP", "ADSK", "AEP", "AMAT", "AMD", "AMGN",
    "AMZN", "APP", "ARM", "ASML", "AVGO", "AXON", "AZN", "BIIB", "BKNG", "BKR",
    "CCEP", "CDNS", "CDW", "CEG", "CHTR", "CMCSA", "COST", "CPRT", "CRWD", "CSCO",
    "CSGP", "CSX", "CTAS", "CTSH", "DASH", "DDOG", "DXCM", "EA", "EXC", "FANG",
    "FAST", "FTNT", "GEHC", "GFS", "GILD", "GOOG", "GOOGL", "HON", "IDXX", "INTC",
    "INTU", "ISRG", "KDP", "KHC", "KLAC", "LIN", "LRCX", "LULU", "MAR", "MCHP",
    "MDLZ", "MELI", "META", "MNST", "MRVL", "MSFT", "MSTR", "MU", "NFLX", "NVDA",
    "NXPI", "ODFL", "ON", "ORLY", "PANW", "PAYX", "PCAR", "PDD", "PEP", "PLTR",
    "PYPL", "QCOM", "REGN", "ROP", "ROST", "SBUX", "SNPS", "TEAM", "TMUS", "TSLA",
    "TTD", "TTWO", "TXN", "VRSK", "VRTX", "WBD", "WDAY", "XEL", "ZS",
]


def nasdaq100_symbols() -> list[str]:
    """Current Nasdaq-100 tickers, scraped, falling back to the snapshot above."""
    try:
        import requests
        from bs4 import BeautifulSoup

        res = requests.get(
            "https://en.wikipedia.org/wiki/Nasdaq-100",
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=20,
        )
        soup = BeautifulSoup(res.text, "html.parser")
        table = soup.find("table", {"id": "constituents"})
        header = [th.get_text(strip=True).lower() for th in table.find("tr").find_all(["th", "td"])]
        col = header.index("ticker") if "ticker" in header else header.index("symbol")
        out = []
        for row in table.find_all("tr")[1:]:
            cells = row.find_all("td")
            if len(cells) > col:
                out.append(cells[col].get_text(strip=True).replace(".", "-"))
        if len(out) >= 90:
            print(f"universe: scraped {len(out)} Nasdaq-100 tickers from Wikipedia")
            return sorted(set(out))
    except Exception as exc:
        print(f"universe: scrape failed ({type(exc).__name__}: {exc})")

    print(f"universe: using the static snapshot ({len(NDX_STATIC)} tickers)")
    return sorted(set(NDX_STATIC))


UNIVERSE = nasdaq100_symbols()
len(UNIVERSE), UNIVERSE[:12]

## 3 · Price history

`data.load_history` is the repo's loader: on-disk CSV cache first, then yfinance.
It returns an empty frame rather than raising when a symbol is unavailable, and it pulls
with `auto_adjust=True` — so QQQ's dividends are already in its price series and the
comparison is total return against total return, not price against price.

**The cache does not expire.** `load_history` reuses `.screen_cache/bt_<SYMBOL>_5y.csv`
whenever it exists, so a second run weeks later replays the old history. Delete those
files to pull fresh data:

```bash
rm .screen_cache/bt_*_5y.csv
```

The first live run fetches ~100 symbols one at a time and takes a few minutes.

If fewer than half the universe comes back — no network, an egress policy that blocks
Yahoo, a throttled session — the notebook switches to `data.synthetic_ohlcv` so the
machinery still runs end to end. **A synthetic run proves the code works and says
nothing whatsoever about the strategy.** The banner below tells you which mode you are in.

In [ ]:
NEEDED = [*UNIVERSE, *CFG.benchmarks, CFG.rs_benchmark]


def load_real_prices(symbols: list[str]) -> dict[str, pd.DataFrame]:
    frames = {}
    for i, sym in enumerate(symbols, 1):
        frame = data_mod.load_history(sym, period=CFG.history_period)
        if not frame.empty:
            frames[sym] = frame
        if i % 25 == 0:
            print(f"  loaded {i}/{len(symbols)} ({len(frames)} with data)")
    return frames


def synthetic_prices(symbols: list[str]) -> dict[str, pd.DataFrame]:
    """Deterministic stand-in history, one distinct path per symbol."""
    synth_start = "2020-10-01"
    n = len(pd.bdate_range(synth_start, END))
    profile = {
        "BOXX": (0.05, 0.006),
        "QQQ": (0.13, 0.19),
        CFG.rs_benchmark: (0.10, 0.15),
    }
    frames = {}
    for seed, sym in enumerate(symbols):
        drift, vol = profile.get(sym, (0.04 + 0.16 * ((seed % 7) / 6.0), 0.20 + 0.05 * (seed % 5)))
        frames[sym] = data_mod.synthetic_ohlcv(
            n=n, seed=1000 + seed, start=synth_start,
            initial_price=40.0 + 3.0 * (seed % 40),
            annual_drift=drift, annual_vol=vol,
        )
    return frames


print(f"loading {len(NEEDED)} symbols...")
PRICES = load_real_prices(NEEDED)
REQUIRED = {*CFG.benchmarks, CFG.rs_benchmark}
SYNTHETIC = len(PRICES) < len(NEEDED) // 2 or any(s not in PRICES for s in REQUIRED)

if SYNTHETIC:
    print("\n" + "!" * 78)
    print("!! SYNTHETIC MODE — real prices are unavailable in this environment.")
    print("!! Every number below is generated from a random walk. It is a test of the")
    print("!! machinery, not a result. Re-run where yfinance is reachable for real figures.")
    print("!" * 78 + "\n")
    PRICES = synthetic_prices(NEEDED)

# One master calendar for marking and execution. QQQ is the exchange calendar the
# strategy actually trades on; a union across 100 symbols would pick up stray dates
# from a single mis-stamped series. Prices are then forward-filled onto it, so a
# symbol missing one bar carries its last close instead of dropping out of the mark
# and putting a hole in the equity curve.
MASTER = PRICES["QQQ"].index if "QQQ" in PRICES else (
    pd.DatetimeIndex(sorted(set().union(*[f.index for f in PRICES.values()]))))

CLOSES = pd.DataFrame({s: f["Close"] for s, f in PRICES.items()}).reindex(MASTER).ffill()
OPENS = pd.DataFrame({s: f["Open"] for s, f in PRICES.items()}).reindex(MASTER).ffill()

CALENDAR = MASTER[(MASTER >= START) & (MASTER <= END)]
print(f"mode: {'SYNTHETIC' if SYNTHETIC else 'live prices'} | symbols with data: {len(PRICES)}")
print(f"history: {CLOSES.index[0].date()} -> {CLOSES.index[-1].date()}")
print(f"backtest window: {CALENDAR[0].date()} -> {CALENDAR[-1].date()} ({len(CALENDAR)} trading days)")

## 4 · Selection — the dashboard's screen, asked point-in-time

`screening.evaluate_ticker` is called directly. The only preparation is truncating each
symbol's history at the as-of date, so `.iloc[-1]` inside the function means "the last
bar that had printed by then".

One shim: the screen resolves each name's sector through yfinance's `.info` endpoint.
Sector is a *label* on the result — `passed` and `daily_annret`, the only two fields the
ranking reads, never touch it — so it is stubbed out rather than firing one network call
per name per rebalance.

In [ ]:
# Sector lookup is a label, not an input to the ranking. See the note above.
screening.yf_info_cached = lambda ticker: {}


def screen_asof(asof: pd.Timestamp, universe: list[str]) -> pd.DataFrame:
    """Run the dashboard's screen using only bars at or before `asof`."""
    bench = CLOSES[CFG.rs_benchmark].loc[:asof].dropna()
    if len(bench) < SCREENING_PARAMS["rs_lookback"]:
        return pd.DataFrame()

    rows = []
    for sym in universe:
        frame = PRICES.get(sym)
        if frame is None:
            continue
        window = frame.loc[:asof].tail(CFG.screen_window)
        if len(window) < SCREENING_PARAMS["rs_lookback"]:
            continue
        try:
            res = screening.evaluate_ticker(sym, window, bench)
        except Exception:
            continue
        if res is not None:
            rows.append(res.__dict__)

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df[df["passed"]].sort_values("daily_annret", ascending=False).reset_index(drop=True)


def select_top_n(asof: pd.Timestamp, universe: list[str], n: int) -> list[str]:
    passed = screen_asof(asof, universe)
    return [] if passed.empty else passed["ticker"].head(n).tolist()


# Smoke test on the first as-of date in the window.
_probe_date = CLOSES.index[CLOSES.index < START][-1]
_probe = screen_asof(_probe_date, UNIVERSE)
print(f"as of {_probe_date.date()}: {len(_probe)} of {len(UNIVERSE)} names passed the screen")
_probe.head(20)[["ticker", "last_close", "daily_annret", "ann_vol", "rs6m_vs_mkt", "within_52w_high_pct"]] if not _probe.empty else "none passed"

## 5 · Sizing — long-only maximum Sharpe

Over the selected names, maximise

$$\text{Sharpe}(w) = \frac{w^{\top}\mu - r_f}{\sqrt{w^{\top}\Sigma w}}
\qquad \text{s.t.}\quad \sum_i w_i = 1,\quad 0 \le w_i \le w_{\max}$$

on annualised trailing estimates. Three things keep it from producing garbage:

* **Shrinkage.** A 252×20 sample covariance is badly conditioned; the raw inverse
  concentrates the whole book in whichever name happened to be quietest. `cov_shrinkage`
  pulls the off-diagonals toward zero.
* **A weight cap.** `max_weight` = 25%, mirroring `config.MAX_POSITION_PCT`.
* **A min-variance fallback.** When no selected name has a trailing excess return above
  $r_f$, the Sharpe objective has no meaningful maximum. Rather than return whatever the
  optimiser lands on, fall back to minimum variance under the same constraints and say so.

The optimiser is multi-start (equal weight, inverse volatility, and random draws) because
SLSQP on this objective is not convex and will happily stop at a local optimum.

In [ ]:
from scipy.optimize import minimize


def _annualised_moments(rets: pd.DataFrame, shrinkage: float) -> tuple[np.ndarray, np.ndarray]:
    mu = rets.mean().to_numpy() * TRADING_DAYS
    sample = rets.cov().to_numpy() * TRADING_DAYS
    target = np.diag(np.diag(sample))
    cov = (1.0 - shrinkage) * sample + shrinkage * target
    # Nudge onto the PSD cone; a sample covariance can come back marginally indefinite.
    cov = (cov + cov.T) / 2.0
    cov += np.eye(len(cov)) * 1e-10
    return mu, cov


def _solve(objective, n: int, max_weight: float, starts: list[np.ndarray]):
    bounds = [(0.0, max_weight)] * n
    constraints = ({"type": "eq", "fun": lambda w: w.sum() - 1.0},)
    best, best_val = None, np.inf
    for w0 in starts:
        try:
            res = minimize(objective, w0, method="SLSQP", bounds=bounds,
                           constraints=constraints, options={"maxiter": 400, "ftol": 1e-10})
        except Exception:
            continue
        if res.success and np.isfinite(res.fun) and res.fun < best_val:
            best, best_val = res.x, res.fun
    return best


def max_sharpe_weights(rets: pd.DataFrame, rf: float, cfg: BacktestSettings, seed: int = 0):
    """Weights, and a note on how they were reached. Returns (Series, str)."""
    n = rets.shape[1]
    if n == 0:
        return pd.Series(dtype=float), "empty"
    if n == 1:
        return pd.Series([1.0], index=rets.columns), "single name"

    max_weight = max(cfg.max_weight, 1.0 / n)  # keep the feasible set non-empty
    mu, cov = _annualised_moments(rets, cfg.cov_shrinkage)

    vols = np.sqrt(np.diag(cov))
    inv_vol = (1.0 / np.where(vols > 0, vols, np.nan))
    inv_vol = np.nan_to_num(inv_vol, nan=0.0)
    inv_vol = inv_vol / inv_vol.sum() if inv_vol.sum() > 0 else np.full(n, 1.0 / n)

    rng = np.random.default_rng(seed)
    starts = [np.full(n, 1.0 / n), np.clip(inv_vol, 0.0, max_weight)]
    starts = [s / s.sum() for s in starts]
    for _ in range(8):
        draw = rng.dirichlet(np.ones(n))
        starts.append(np.clip(draw, 0.0, max_weight) / np.clip(draw, 0.0, max_weight).sum())

    def neg_variance(w):
        return float(w @ cov @ w)

    if mu.max() <= rf:
        w = _solve(neg_variance, n, max_weight, starts)
        note = "min-variance fallback (no name above rf)"
    else:
        def neg_sharpe(w):
            excess = float(w @ mu) - rf
            vol = np.sqrt(max(float(w @ cov @ w), 1e-18))
            return -excess / vol
        w = _solve(neg_sharpe, n, max_weight, starts)
        note = "max-sharpe"
        if w is None:
            w = _solve(neg_variance, n, max_weight, starts)
            note = "min-variance fallback (optimiser failed)"

    if w is None:
        return pd.Series(np.full(n, 1.0 / n), index=rets.columns), "equal weight fallback"

    w = np.clip(w, 0.0, None)
    w = w / w.sum() if w.sum() > 0 else np.full(n, 1.0 / n)
    return pd.Series(w, index=rets.columns), note


def risk_free_rate(asof: pd.Timestamp, cfg: BacktestSettings) -> float:
    """Annualised trailing return of the risk-free proxy, or 0.0 without one."""
    if not cfg.rf_proxy or cfg.rf_proxy not in CLOSES.columns:
        return 0.0
    series = CLOSES[cfg.rf_proxy].loc[:asof].dropna().tail(cfg.cov_lookback)
    if len(series) < cfg.cov_min_bars:
        return 0.0
    return float(series.pct_change().dropna().mean() * TRADING_DAYS)

## 6 · The back-test

The month boundary is the only place anything trades:

* **Signal date** — the last close *before* the first trading day of the month. The screen
  and the covariance both stop there.
* **Execution** — the open of the first trading day of the month, at
  `CostModel.fill_price`, with commission on the notional.
* **Rebalance** — the roll from last month's weights to this month's happens in that one
  trade: names that dropped out are sold, survivors are adjusted to their new weight.
  Holding to the end of the month and re-entering is the same thing with a gap in
  exposure, so the position is carried straight across the boundary.
* **Marking** — equity is marked at the close every day in between; nothing trades.

Nothing in the loop reads a bar later than the signal date when choosing what to hold,
which is the property the repo's `tests/test_causality.py` asserts for the breakout engine.

In [ ]:
def rebalance_dates(calendar: pd.DatetimeIndex) -> list[pd.Timestamp]:
    """First trading day of each month inside the window."""
    frame = pd.Series(calendar, index=calendar)
    return [g.iloc[0] for _, g in frame.groupby(calendar.to_period("M"))]


def target_weights(asof: pd.Timestamp, cfg: BacktestSettings):
    """(weights, picks, note) for the month beginning after `asof`."""
    picks = select_top_n(asof, UNIVERSE, cfg.top_n)
    if not picks:
        return pd.Series(dtype=float), [], "no name passed the screen — hold cash"

    window = CLOSES[picks].loc[:asof].tail(cfg.cov_lookback + 1)
    rets = window.pct_change().dropna(how="all")
    rets = rets.dropna(axis=1, thresh=cfg.cov_min_bars).dropna()
    if rets.shape[1] == 0 or len(rets) < cfg.cov_min_bars:
        return pd.Series(dtype=float), picks, "insufficient return history — hold cash"

    weights, note = max_sharpe_weights(rets, risk_free_rate(asof, cfg), cfg)
    return weights, picks, note


def run_strategy(cfg: BacktestSettings):
    calendar = CALENDAR
    rebals = set(rebalance_dates(calendar))

    cash = cfg.principal
    shares: dict[str, float] = {}
    equity_rows, trade_rows, plan_rows = [], [], []

    for day in calendar:
        opens, closes = OPENS.loc[day], CLOSES.loc[day]

        if day in rebals:
            prior = CLOSES.index[CLOSES.index < day]
            asof = prior[-1]
            weights, picks, note = target_weights(asof, cfg)

            mark = {s: float(opens.get(s, np.nan)) for s in shares}
            equity = cash + sum(q * mark[s] for s, q in shares.items() if np.isfinite(mark.get(s, np.nan)))
            investable = equity * (1.0 - cfg.cost_buffer)

            targets = {s: float(w) * investable for s, w in weights.items()
                       if np.isfinite(opens.get(s, np.nan)) and opens.get(s, 0) > 0}

            for sym in sorted(set(shares) | set(targets)):
                ref = float(opens.get(sym, np.nan))
                if not np.isfinite(ref) or ref <= 0:
                    continue
                held = shares.get(sym, 0.0)
                want = targets.get(sym, 0.0) / ref
                delta = want - held
                if abs(delta * ref) < 1.0:      # skip dust
                    continue
                side = Action.BUY if delta > 0 else Action.SELL
                fill = cfg.costs.fill_price(ref, side)
                notional = delta * fill
                fee = cfg.costs.costs(notional)
                cash -= notional + fee
                shares[sym] = held + delta
                trade_rows.append({"date": day, "symbol": sym, "side": side.value,
                                   "shares": delta, "fill": fill, "notional": notional, "cost": fee})

            shares = {s: q for s, q in shares.items() if abs(q) > 1e-9}
            plan_rows.append({"rebalance": day, "signal_date": asof, "n_passed": len(picks),
                              "n_held": len(shares), "note": note,
                              "weights": weights.round(4).to_dict()})

        marked = sum(q * float(closes.get(s, np.nan)) for s, q in shares.items()
                     if np.isfinite(closes.get(s, np.nan)))
        equity_rows.append({"date": day, "equity": cash + marked, "cash": cash, "n_positions": len(shares)})

    equity = pd.DataFrame(equity_rows).set_index("date")
    return equity, pd.DataFrame(trade_rows), pd.DataFrame(plan_rows)


def buy_and_hold(symbol: str, cfg: BacktestSettings) -> pd.Series:
    """Principal into one symbol at the first open of the window, marked to close."""
    first = CALENDAR[0]
    ref = float(OPENS.loc[first, symbol])
    fill = cfg.costs.fill_price(ref, Action.BUY)
    qty = (cfg.principal - cfg.costs.costs(cfg.principal)) / fill
    return (CLOSES.loc[CALENDAR, symbol] * qty).rename(symbol)

In [ ]:
EQUITY, TRADES, PLAN = run_strategy(CFG)
CURVES = pd.DataFrame({"Strategy": EQUITY["equity"]})
for sym in CFG.benchmarks:
    if sym in CLOSES.columns:
        CURVES[sym] = buy_and_hold(sym, CFG)

print(f"rebalances: {len(PLAN)} | fills: {len(TRADES)} | "
      f"commission: ${TRADES['cost'].sum():,.2f} (slippage is inside every fill price) | "
      f"lowest cash balance: ${EQUITY['cash'].min():,.2f}")
CURVES.tail()

## 7 · Results

`equity_metrics` mirrors `backtest._compute_metrics` — same Sharpe (rf = 0, on daily
equity returns), same CAGR, same drawdown definition — so a figure here is comparable to
one printed by `backtest.py`.

In [ ]:
def equity_metrics(equity: pd.Series) -> dict:
    """Same definitions as backtest._compute_metrics, on an equity curve alone."""
    equity = equity.dropna()
    start_val, end_val = float(equity.iloc[0]), float(equity.iloc[-1])
    rets = equity.pct_change().dropna()
    std = float(rets.std())
    years = len(equity) / TRADING_DAYS
    drawdown = equity / equity.cummax() - 1.0
    max_dd = float(drawdown.min())
    cagr = (end_val / start_val) ** (1 / years) - 1 if years > 0 and start_val > 0 else 0.0
    return {
        "Ending equity": end_val,
        "Profit": end_val - start_val,
        "Total return": end_val / start_val - 1 if start_val > 0 else 0.0,
        "CAGR": cagr,
        "Ann. vol": std * np.sqrt(TRADING_DAYS) if std > 0 else 0.0,
        "Sharpe (rf=0)": (float(rets.mean()) / std) * np.sqrt(TRADING_DAYS) if std > 0 else 0.0,
        "Max drawdown": max_dd,
        "Calmar": cagr / abs(max_dd) if max_dd < 0 else 0.0,
        "Best day": float(rets.max()),
        "Worst day": float(rets.min()),
    }


SUMMARY = pd.DataFrame({name: equity_metrics(CURVES[name]) for name in CURVES.columns}).T
PCT = ["Total return", "CAGR", "Ann. vol", "Max drawdown", "Best day", "Worst day"]

display_summary = SUMMARY.copy()
for col in PCT:
    display_summary[col] = display_summary[col].map(lambda v: f"{v:.2%}")
for col in ["Ending equity", "Profit"]:
    display_summary[col] = display_summary[col].map(lambda v: f"${v:,.0f}")
for col in ["Sharpe (rf=0)", "Calmar"]:
    display_summary[col] = display_summary[col].map(lambda v: f"{v:.2f}")

print(f"${CFG.principal:,.0f} from {CALENDAR[0].date()} to {CALENDAR[-1].date()}"
      + ("   [SYNTHETIC DATA]" if SYNTHETIC else ""))
display_summary

In [ ]:
best = SUMMARY["Ending equity"].idxmax()
strat = SUMMARY.loc["Strategy"]
lines = [f"Winner: {best}  (${SUMMARY.loc[best, 'Ending equity']:,.0f})", ""]
for name in CURVES.columns:
    row = SUMMARY.loc[name]
    lines.append(f"  {name:<10} ${row['Ending equity']:>10,.0f}   "
                 f"profit ${row['Profit']:>9,.0f}   {row['Total return']:>8.2%}   "
                 f"Sharpe {row['Sharpe (rf=0)']:>5.2f}   maxDD {row['Max drawdown']:>7.2%}")
lines.append("")
for name in CURVES.columns:
    if name == "Strategy":
        continue
    diff = strat["Profit"] - SUMMARY.loc[name, "Profit"]
    verb = "ahead of" if diff >= 0 else "behind"
    lines.append(f"  Strategy is ${abs(diff):,.0f} {verb} buy-and-hold {name}.")
print("\n".join(lines))

### Equity curves

One axis, one currency, three series — the comparison the question actually asks for.
Each line is labelled at its right-hand end as well as in the legend, so the series are
never identified by colour alone.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# Categorical slots 1-3 of the validated default palette (blue / orange / aqua).
SERIES_COLORS = {"Strategy": "#2a78d6", "QQQ": "#eb6834", "BOXX": "#1baf7a"}
INK, INK_MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
USD = FuncFormatter(lambda v, _: f"${v:,.0f}")


def end_labels(ax, entries, min_gap=0.07):
    """Right-edge series labels, nudged apart when two lines finish close together.

    `entries` is [(text, x, y, color)]. A label pushed off its own line gets a thin
    leader in the series colour, so the pairing stays unambiguous.
    """
    lo, hi = ax.get_ylim()
    gap = min_gap * (hi - lo)
    ordered = sorted(entries, key=lambda e: e[2])
    placed, last = [], None
    for _, _, y, _ in ordered:
        last = y if last is None else max(y, last + gap)
        placed.append(last)
    for target, (text, x, y, color) in zip(placed, ordered):
        ax.annotate(text, (x, target), color=INK, fontsize=9, va="center", ha="left",
                    xytext=(8, 0), textcoords="offset points", annotation_clip=False)
        if abs(target - y) > gap * 0.2:
            ax.plot([x, x], [y, target], color=color, linewidth=0.8, alpha=0.55, clip_on=False)


def style_axes(ax, title, ylabel, formatter=USD):
    ax.set_title(title, color=INK, fontsize=13, pad=12, loc="left")
    ax.set_ylabel(ylabel, color=INK_MUTED, fontsize=10)
    ax.yaxis.set_major_formatter(formatter)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=INK_MUTED, labelsize=9, length=0)


fig, ax = plt.subplots(figsize=(11, 5.5), dpi=130)
labels = []
for name in CURVES.columns:
    series = CURVES[name].dropna()
    color = SERIES_COLORS.get(name, "#eda100")
    ax.plot(series.index, series.values, color=color, linewidth=2.0, label=name,
            solid_capstyle="round")
    labels.append((f"{name} ${series.iloc[-1]:,.0f}", series.index[-1], series.iloc[-1], color))

ax.axhline(CFG.principal, color=INK_MUTED, linewidth=1.0, linestyle=(0, (4, 4)), zorder=0)
style_axes(ax, f"${CFG.principal:,.0f} invested {CALENDAR[0].date()} — "
               + ("SYNTHETIC DATA" if SYNTHETIC else "portfolio value"), "Portfolio value")
ax.legend(frameon=False, loc="upper left", fontsize=9, labelcolor=INK_MUTED)
ax.margins(x=0.10)
end_labels(ax, labels)
fig.tight_layout()
plt.show()

### Drawdown

Peak-to-trough on the same three curves. This is the half of the comparison a total-return
number hides: BOXX is here to show what almost no drawdown looks like next to the other two.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2), dpi=130)
for name in CURVES.columns:
    series = CURVES[name].dropna()
    dd = (series / series.cummax() - 1.0) * 100
    color = SERIES_COLORS.get(name, "#eda100")
    ax.plot(dd.index, dd.values, color=color, linewidth=2.0, label=name)
    # Label the trough rather than the right edge: it is the number the chart is
    # about, and three curves that all finish near 0% would stack their labels.
    trough = dd.idxmin()
    ax.plot([trough], [dd.min()], marker="o", markersize=6, color=color,
            markeredgecolor="#fcfcfb", markeredgewidth=1.5, zorder=5)
    ax.annotate(f"{name} {dd.min():.1f}%", (trough, dd.min()), color=INK, fontsize=9,
                va="top", ha="center", xytext=(0, -8), textcoords="offset points")

style_axes(ax, "Drawdown from running peak", "Drawdown",
           FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.axhline(0, color=GRID, linewidth=1.0)
ax.legend(frameon=False, loc="lower left", fontsize=9, labelcolor=INK_MUTED)
ax.margins(x=0.04, y=0.14)
fig.tight_layout()
plt.show()

### Month by month

Where the total came from. Grouped bars, one group per calendar month, with a 2px surface
gap between adjacent fills so neighbouring bars stay distinct without a border colour.

In [ ]:
# Base the first month on the principal, not on day one's close, so the month the
# money went in carries the cost of getting in.
opening = pd.DataFrame([[CFG.principal] * CURVES.shape[1]], columns=CURVES.columns,
                       index=[CALENDAR[0] - pd.Timedelta(days=1)])
monthly = pd.concat([opening, CURVES.resample("ME").last()]).pct_change().dropna(how="all") * 100
monthly.index = monthly.index.strftime("%Y-%m")

fig, ax = plt.subplots(figsize=(11, 4.6), dpi=130)
names = list(CURVES.columns)
width = 0.8 / len(names)
x = np.arange(len(monthly))
for i, name in enumerate(names):
    ax.bar(x + i * width - 0.4 + width / 2, monthly[name].values, width * 0.94,
           color=SERIES_COLORS.get(name, "#eda100"), label=name,
           edgecolor="#fcfcfb", linewidth=1.0)

style_axes(ax, "Monthly return", "Return", FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.axhline(0, color=INK_MUTED, linewidth=1.0)
ax.set_xticks(x)
ax.set_xticklabels(monthly.index, rotation=45, ha="right")
ax.legend(frameon=False, loc="best", fontsize=9, labelcolor=INK_MUTED)
fig.tight_layout()
plt.show()

monthly.round(2)

### What it held, and why

`PLAN` is the audit trail: one row per rebalance, with the signal date, how many names
cleared the screen, and how the weights were reached. A `min-variance fallback` note means
the screen's survivors had no trailing excess return over the risk-free proxy that month.

In [ ]:
plan_view = PLAN[["rebalance", "signal_date", "n_passed", "n_held", "note"]].copy()
plan_view["rebalance"] = plan_view["rebalance"].dt.date
plan_view["signal_date"] = plan_view["signal_date"].dt.date
plan_view["top holdings"] = [
    ", ".join(f"{s} {w:.0%}" for s, w in sorted(d.items(), key=lambda kv: -kv[1])[:6]
              if w >= 0.005) or "—"
    for d in PLAN["weights"]
]
plan_view

In [ ]:
# Full weight matrix: rebalance date x symbol. Blank = not held that month.
weight_matrix = pd.DataFrame(list(PLAN["weights"]), index=PLAN["rebalance"].dt.date)
weight_matrix = weight_matrix.reindex(sorted(weight_matrix.columns), axis=1)
print(f"{weight_matrix.shape[1]} distinct names held across {len(weight_matrix)} rebalances")
(weight_matrix * 100).round(1).fillna("")

In [ ]:
# Every fill, if you want to check the cost model or reconcile a month.
trades_view = TRADES.copy()
if not trades_view.empty:
    trades_view["date"] = trades_view["date"].dt.date
    trades_view = trades_view.round({"shares": 3, "fill": 2, "notional": 2, "cost": 2})
print(f"{len(trades_view)} fills, ${TRADES['cost'].sum():,.2f} in commission, "
      f"{TRADES['notional'].abs().sum() / CFG.principal:.1f}x principal turned over")
trades_view.head(40)

## 8 · Reading this honestly

* **The window is one regime.** June 2025 onward is fifteen months. A Sharpe computed on
  it has an enormous standard error; the ranking between the strategy and QQQ could
  plausibly invert on a different fifteen months. Treat the table as a description of what
  happened, not an estimate of what will.
* **Max-Sharpe weights are the fragile part.** Mean-variance optimisation is famously
  sensitive to the mean estimate, and a trailing 252-day mean is a weak one. The shrinkage
  and the 25% cap are there to blunt that, and they are also the reason the result is not a
  pure statement about the objective.
* **The screen already selects on momentum**, and `daily_annret` ranks on it again. Feeding
  those trailing returns into the optimiser as $\mu$ compounds the same bet three times.
  Setting `rf_proxy=None` and `cov_shrinkage=1.0` collapses the sizing toward
  risk-parity-ish weights, which is a useful contrast run.
* **BOXX is the right kind of baseline** — it is the "did taking any equity risk at all pay"
  question. QQQ is the "did the selection beat just owning the index" question. They answer
  different things and the strategy can lose to one and beat the other.

### Things worth changing and re-running

```python
CFG = BacktestSettings(top_n=10)                       # concentration
CFG = BacktestSettings(max_weight=1.0)                 # uncapped max-Sharpe
CFG = BacktestSettings(cov_shrinkage=0.0)              # raw sample covariance
CFG = BacktestSettings(start="2023-01-01")             # a longer, still short, window
CFG = BacktestSettings(costs=CostModel(slippage_bps=20, commission_bps=5))
```

Re-run from the configuration cell down after changing it.